<a href="https://colab.research.google.com/github/Takayoshi-code/nanoGPT-class/blob/master/for_class3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 【STEP1】 Pre-Trained Model（Q４圧縮版）をgithubからダウンロードする。427MBありますので若干時間がかかります。

In [ ]:
# @title
!wget -O /content/nanogpt-670m-base-q4.gguf \
"https://github.com/Takayoshi-code/nanoGPT-class/releases/download/nanoGPT670MQ4models/nanogpt-670m-base-q4.gguf"

##SKIP対象　量子化の様子の確認１

In [ ]:
# @title
%pip install gguf
from gguf import GGUFReader

reader = GGUFReader("nanogpt-670m-base-q4.gguf")

for field in reader.fields.values():
    if field.name == "general.file_type":
        print(field.contents())

##SKIP対象　量子化の様子の確認２

In [ ]:
# @title
from gguf import GGUFReader
from collections import Counter

path = "nanogpt-670m-base-q4.gguf"

reader = GGUFReader(path)

# 各テンソルの量子化方式を集計
types = Counter(t.tensor_type.name for t in reader.tensors)

print("=== 量子化方式別のテンソル数 ===")
for name, count in types.items():
    print(f"{name}: {count}")

print("\n=== 各テンソルの詳細 ===")
for t in reader.tensors:
    print(f"{t.name}: {t.tensor_type.name}")

## 【STEP2】SFT model(Q4圧縮版）をgithubからダウンロードする。427MBありますので若干時間がかかります。

In [ ]:
# @title
!wget -O /content/nanogpt-670m-sft500-q4.gguf \
"https://github.com/Takayoshi-code/nanoGPT-class/releases/download/nanoGPT670MQ4models/nanogpt-670m-sft500-q4.gguf"

## 【STEP3】 llama-cpp-pythonをインストールする。llama.cpp はLLMを実行するC/C++ベースの推論エンジンで、llama-cpp-python はそれをPythonから操作するためのライブラリです。

In [ ]:
# @title
!pip install llama-cpp-python \
  --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cpu

## 【STEP4】　670MサイズのLLMをロードします。

## [STEP4-1] 670M Pre-Trained Modelをllamaにロードする。

In [5]:
# @title
from llama_cpp import Llama

llm = Llama(
    model_path="/content/nanogpt-670m-base-q4.gguf",
    n_ctx=2048,
    n_gpu_layers=0,
    n_threads=4,
    verbose=False
)

## 【STEP4-2】 670M Pre-Trained Modelで文書生成を試す。サンプリング

In [ ]:
# @title
import textwrap

for i in range(5):

    result = llm(
        "東京タワーは",
        max_tokens=200,
        temperature=0.7,
        top_k=50,
        top_p=0.9,
        repeat_penalty=1.15
    )

    print(f"\n--- sample {i+1} ---")

    text = result["choices"][0]["text"]

    print(textwrap.fill(
        text,
        width=80,
        break_long_words=True,
        break_on_hyphens=False
    ))

##【STEP4-3】 670M Pre-Trained Modelで文書生成を試す。greedy

In [ ]:
# @title
import textwrap

result = llm(
    "東京タワーは",
    max_tokens=200,
    temperature=0.0,
    repeat_penalty=1.15
)

text = result["choices"][0]["text"]

print(textwrap.fill(
    text,
    width=80,
    break_long_words=True,
    break_on_hyphens=False
))

## 【STEP4-4】 670M SFT modelをロードする。

In [8]:
# @title
from llama_cpp import Llama

sft_model_path = "/content/nanogpt-670m-sft500-q4.gguf"

llm_sft = Llama(
    model_path=sft_model_path,
    n_ctx=2048,
    n_gpu_layers=0,
    n_threads=4,
    verbose=False
)

## 【STEP4-5】 670M SFT modelで質疑応答を試す。サンプリング

In [ ]:
# @title
prompt = """<|user|>
東京ドームを本拠地にしている野球チームは？
<|assistant|>
"""
for i in range(5):
    result = llm_sft(
        prompt,
        max_tokens=100,
        temperature=0.7,
        top_k=50,
        top_p=0.9,
        repeat_penalty=1.15
    )

    print(f"\n--- sample {i+1} ---")
    print(result["choices"][0]["text"])

## 【STEP4-6】 670M SFT modelで質疑応答を試す。greedy

In [ ]:
# @title
prompt = """<|user|>
東京ドームを本拠地にしている野球チームは？
<|assistant|>
"""
result = llm_sft(
    prompt,
    max_tokens=100,
    temperature=0.0,
    repeat_penalty=1.15
)

print(result["choices"][0]["text"])

## 【STEP6】 RAG(Retrieval-Augmented Generation）検索拡張生成機能を試す。LLMの内部内の知識が不足した場合に外部情報に頼って回答を得る方法。

## 【STEP6-1】 RAG用のtext fileを作成する。

In [11]:
# @title
knowledge = """富士山の標高は3776メートルである。静岡県と山梨県にまたがる日本最高峰の山である。

東京タワーの高さは333メートルである。東京都港区芝公園に位置する電波塔である。

東京スカイツリーの高さは634メートルである。

日本の首都は東京である。国会議事堂や中央省庁が置かれている。

筑波山の標高は877メートルである。茨城県つくば市に位置する山である。

太宰治は日本の小説家である。代表作には『人間失格』『斜陽』『走れメロス』がある。

モンブランの麓の町はシャモニーである。

エベレストは世界最高峰で高さは8,848.86メートルです。

マイケルジャクソンの代表作はBillie Jeanである。

応用実践第３Cは今年度はかなりチャレンジングな内容である。

100m走の世界記録はジャマイカのウサイン・ボルトが2009年に出した9秒58です。

うちの猫の名前はサバです。 """

with open("knowledge.txt", "w", encoding="utf-8") as f:
    f.write(knowledge)

documents = [
    s.strip()
    for s in knowledge.split("\n\n")
    if s.strip()
]

print("文書数:", len(documents))

文書数: 12


## 【STEP6-2】 TF-IDFによる検索機能を確認する。

In [ ]:
# @title
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=(2, 3)
)

X = vectorizer.fit_transform(documents)

def retrieve(question):
    q = vectorizer.transform([question])
    scores = cosine_similarity(q, X)[0]
    i = scores.argmax()
    return documents[i], scores[i]

questions = [
    "筑波山の高さは？",
    "東京タワーはどのくらいの高さ？",
    "東京スカイツリーはどのくらいの高さ？",
    "モンブランで有名な町は？",
    "マイケルジャクソンの有名な曲は？",
    "エベレストの標高は？",
    "応用実践第３Cの内容は？",
    "100m走の世界記録は？",
    "うちに遊びに来る猫の名前は？"
]

for q in questions:
    doc, score = retrieve(q)
    print("\n質問:", q)
    print("検索結果:", doc)
    print("類似度:", round(float(score), 3))

## 【STEP6-3】 RAGを使ったLLMの実行　文書検索にはTF-IDFを用いる。

In [ ]:
# @title

from llama_cpp import Llama
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import os

# ============================================================
# 1. 基本設定
# ============================================================

MODEL_PATH = "/content/nanogpt-670m-sft500-q4.gguf"
KNOWLEDGE_PATH = "/content/knowledge.txt"

TOP_K = 1
MIN_SCORE = 0.30

MAX_TOKENS = 60
TEMPERATURE = 0.0
REPEAT_PENALTY = 1.15

# RAG用プロンプトの選択
# A: 従来の詳しい指示
# B: 短い抽出指示
# C: 参考資料と質問を明示（推奨）
PROMPT_TYPE = "C"

# ============================================================
# 2. GGUF SFTモデルをロード
# ============================================================

if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(
        f"モデルファイルがありません: {MODEL_PATH}"
    )

llm = Llama(
    model_path=MODEL_PATH,
    n_ctx=2048,
    n_threads=4,
    n_gpu_layers=0,
    verbose=False
)

print("SFTモデルのロード完了")

# ============================================================
# 3. RAG用知識データベース
#    knowledge.txtは空行で文書を区切る
# ============================================================

if not os.path.exists(KNOWLEDGE_PATH):
    raise FileNotFoundError(
        f"知識ファイルがありません: {KNOWLEDGE_PATH}"
    )

with open(KNOWLEDGE_PATH, "r", encoding="utf-8") as f:
    documents = [
        x.strip()
        for x in f.read().split("\n\n")
        if x.strip()
    ]

if not documents:
    raise ValueError("knowledge.txtに文書がありません")

print("RAG文書数:", len(documents))

# ============================================================
# 4. TF-IDF検索インデックスを作成
# ============================================================

vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=(2, 3)
)

document_vectors = vectorizer.fit_transform(documents)

print("TF-IDFインデックス作成完了")

# ============================================================
# 5. 関連文書検索
# ============================================================

def retrieve(question, top_k=TOP_K, min_score=MIN_SCORE):

    question_vector = vectorizer.transform([question])

    scores = cosine_similarity(
        question_vector,
        document_vectors
    )[0]

    indices = np.argsort(scores)[::-1][:top_k]

    results = []

    for i in indices:
        score = float(scores[i])

        if score >= min_score:
            results.append((documents[i], score))

    return results

# ============================================================
# 6. SFTモデルによる回答生成
# ============================================================

def generate_answer(user_message):

    prompt = (
        "<|user|>\n"
        + user_message
        + "\n<|assistant|>\n"
    )
 ##   print(prompt)
    result = llm(
        prompt,
        max_tokens=MAX_TOKENS,
        temperature=TEMPERATURE,
        repeat_penalty=REPEAT_PENALTY,
        stop=[
            "<|endoftext|>",
            "<|user|>",
            "<|assistant|>",
            "\n質問：",
            "\n質問:",
            "\n回答：",
            "\n回答:"
        ],
        echo=False
    )

    answer = result["choices"][0]["text"].strip()

    for marker in [
        "\n質問：",
        "\n質問:",
        "\n回答：",
        "\n回答:"
    ]:
        if marker in answer:
            answer = answer.split(marker)[0].strip()

    if not answer:
        answer = "分かりません"

    return answer

# ============================================================
# 7. RAG用プロンプト
# ============================================================

def make_rag_prompt(context, question, prompt_type="C"):

    # A: 従来の詳しい指示
    if prompt_type == "A":

        return f"""次の参考資料だけに基づいて質問に簡潔に答えてください。
資料に答えがない場合は「分かりません」と答えてください。

参考資料：
{context}

質問：
{question}"""

    # B: 簡潔な抽出指示
    elif prompt_type == "B":

        return f"""次の文章から答えを抜き出してください。

{context}

{question}"""

    # C: 参考資料と質問を明示
    elif prompt_type == "C":

        return f"""参考資料：
{context}

質問：{question}

参考資料から答えを抜き出して、短く答えてください。"""

    else:
        raise ValueError(
            "PROMPT_TYPEはA、B、Cから選んでください"
        )

# ============================================================
# 8. RAGなし / Fallback付きRAG
# ============================================================

def ask_sft(question, use_rag=True, prompt_type=PROMPT_TYPE):

    # --------------------------------------------------------
    # RAGなし
    # --------------------------------------------------------

    if not use_rag:

        answer = generate_answer(question)

        return answer, [], "RAGなし"

    # --------------------------------------------------------
    # 関連文書検索
    # --------------------------------------------------------

    retrieved = retrieve(
        question,
        top_k=TOP_K,
        min_score=MIN_SCORE
    )

    # --------------------------------------------------------
    # Fallback：検索結果がない場合は内部知識を使用
    # --------------------------------------------------------

    if not retrieved:

        answer = generate_answer(question)

        return answer, [], "Fallback（内部知識）"

    # --------------------------------------------------------
    # 検索結果がある場合はRAG
    # --------------------------------------------------------

    context = "\n".join(
        doc for doc, score in retrieved
    )

    user_message = make_rag_prompt(
        context,
        question,
        prompt_type=prompt_type
    )

    answer = generate_answer(user_message)

    return answer, retrieved, "RAG（外部知識）"

# ============================================================
# 9. 評価用質問
# ============================================================

questions = [
    "富士山の標高は？",
    "東京タワーの高さは？",
    "東京スカイツリーの高さは？",
    "日本の首都は？",
    "筑波山の標高は？",
    "太宰治の代表作は？",
    "モンブランの麓の町は？",
    "100m走の世界記録は？",
    "マイケルジャクソンの代表作は？",
    "うちに遊びに来る猫の名前は？"
]

# ============================================================
# 10. RAGなし / Fallback付きRAGの比較
# ============================================================

print("\n" + "=" * 70)
print("RAG評価開始")
print("PROMPT_TYPE:", PROMPT_TYPE)
print("MIN_SCORE:", MIN_SCORE)
print("=" * 70)

for question in questions:

    print("=" * 70)
    print("質問:", question)

    # RAGなし
    answer_no_rag, _, _ = ask_sft(
        question,
        use_rag=False
    )

    print("\n【RAGなし】")
    print(answer_no_rag)

    # Fallback付きRAG
    answer_rag, retrieved, mode = ask_sft(
        question,
        use_rag=True,
        prompt_type=PROMPT_TYPE
    )

    print("\n【検索された文章】")

    if retrieved:
        for doc, score in retrieved:
            print(f"score={score:.3f}: {doc}")
    else:
        print("該当する文書なし")

    print("\n【回答方式】")
    print(mode)

    print("\n【RAGあり（Fallback付き）】")
    print(answer_rag)

print("\n全質問の評価終了")


##【STEP6-4】RAG 文書検索にがEmbedding方式を用いる。

##【STEP6-4-0】 Embedding方式の文書検索のライブラリーをインストールする。

In [ ]:
# @title
!pip install -q sentence-transformers llama-cpp-python scikit-learn

##【STEP6-4-1】 RAGの実行　Embedding方式

In [ ]:
# @title

from llama_cpp import Llama
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import os

# ============================================================
# 1. 基本設定
# ============================================================

MODEL_PATH = "/content/nanogpt-670m-sft500-q4.gguf"
KNOWLEDGE_PATH = "/content/knowledge.txt"

# Embeddingモデル（日本語対応、384次元）
EMBEDDING_MODEL = (
    "sentence-transformers/"
    "paraphrase-multilingual-MiniLM-L12-v2"
)

TOP_K = 1

# TF-IDFとはスコア分布が異なるため、実験用の初期値
MIN_SCORE = 0.30

MAX_TOKENS = 60
TEMPERATURE = 0.0
REPEAT_PENALTY = 1.15

# RAG用プロンプト
# A: 詳しい指示
# B: 短い抽出指示
# C: 参考資料と質問を明示
PROMPT_TYPE = "C"

# ============================================================
# 2. GGUF SFTモデルをロード
# ============================================================

if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(
        f"モデルファイルがありません: {MODEL_PATH}"
    )

llm = Llama(
    model_path=MODEL_PATH,
    n_ctx=2048,
    n_threads=4,
    n_gpu_layers=0,
    verbose=False
)

print("SFTモデルのロード完了")

# ============================================================
# 3. RAG用知識データベース
#    knowledge.txtは空行で文書を区切る
# ============================================================

if not os.path.exists(KNOWLEDGE_PATH):
    raise FileNotFoundError(
        f"知識ファイルがありません: {KNOWLEDGE_PATH}"
    )

with open(KNOWLEDGE_PATH, "r", encoding="utf-8") as f:
    documents = [
        x.strip()
        for x in f.read().split("\n\n")
        if x.strip()
    ]

if not documents:
    raise ValueError("knowledge.txtに文書がありません")

print("RAG文書数:", len(documents))

# ============================================================
# 4. Embeddingモデルのロード
# ============================================================

print("\nEmbeddingモデルをロード中...")

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL
)

print("Embeddingモデルのロード完了")

# ============================================================
# 5. 文書Embeddingインデックスを作成
# ============================================================

print("\n文書をEmbeddingベクトルに変換中...")

document_vectors = embedding_model.encode(
    documents,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Embeddingインデックス作成完了")
print("Embedding行列の形状:", document_vectors.shape)
print("Embedding次元数:", document_vectors.shape[1])

# ============================================================
# 6. 関連文書検索
# ============================================================

def retrieve(question, top_k=TOP_K, min_score=MIN_SCORE):

    # 質問をEmbeddingベクトルに変換
    question_vector = embedding_model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    # コサイン類似度を計算
    scores = cosine_similarity(
        question_vector,
        document_vectors
    )[0]

    # 類似度の高い順に文書を取得
    indices = np.argsort(scores)[::-1][:top_k]

    results = []

    for i in indices:
        score = float(scores[i])

        if score >= min_score:
            results.append((documents[i], score))

    return results

# ============================================================
# 7. SFTモデルによる回答生成
# ============================================================

def generate_answer(user_message):

    prompt = (
        "<|user|>\n"
        + user_message
        + "\n<|assistant|>\n"
    )

    result = llm(
        prompt,
        max_tokens=MAX_TOKENS,
        temperature=TEMPERATURE,
        repeat_penalty=REPEAT_PENALTY,
        stop=[
            "<|endoftext|>",
            "<|user|>",
            "<|assistant|>",
            "\n質問：",
            "\n質問:",
            "\n回答：",
            "\n回答:"
        ],
        echo=False
    )

    answer = result["choices"][0]["text"].strip()

    for marker in [
        "\n質問：",
        "\n質問:",
        "\n回答：",
        "\n回答:"
    ]:
        if marker in answer:
            answer = answer.split(marker)[0].strip()

    if not answer:
        answer = "分かりません"

    return answer

# ============================================================
# 8. RAG用プロンプト
# ============================================================

def make_rag_prompt(context, question, prompt_type="C"):

    # A: 詳しい指示
    if prompt_type == "A":

        return f"""次の参考資料だけに基づいて質問に簡潔に答えてください。
資料に答えがない場合は「分かりません」と答えてください。

参考資料：
{context}

質問：
{question}"""

    # B: 短い抽出指示
    elif prompt_type == "B":

        return f"""次の文章から答えを抜き出してください。

{context}

{question}"""

    # C: 参考資料と質問を明示
    elif prompt_type == "C":

        return f"""参考資料：
{context}

質問：{question}

参考資料から答えを抜き出して、短く答えてください。"""

    else:
        raise ValueError(
            "PROMPT_TYPEはA、B、Cから選んでください"
        )

# ============================================================
# 9. RAGなし / Fallback付きRAG
# ============================================================

def ask_sft(question, use_rag=True, prompt_type=PROMPT_TYPE):

    # --------------------------------------------------------
    # RAGなし
    # --------------------------------------------------------

    if not use_rag:

        answer = generate_answer(question)

        return answer, [], "RAGなし"

    # --------------------------------------------------------
    # Embeddingによる関連文書検索
    # --------------------------------------------------------

    retrieved = retrieve(
        question,
        top_k=TOP_K,
        min_score=MIN_SCORE
    )

    # --------------------------------------------------------
    # Fallback：検索結果がない場合は内部知識を使用
    # --------------------------------------------------------

    if not retrieved:

        answer = generate_answer(question)

        return answer, [], "Fallback（内部知識）"

    # --------------------------------------------------------
    # 関連文書がある場合はRAG
    # --------------------------------------------------------

    context = "\n".join(
        doc for doc, score in retrieved
    )

    user_message = make_rag_prompt(
        context,
        question,
        prompt_type=prompt_type
    )

    answer = generate_answer(user_message)

    return answer, retrieved, "RAG（Embedding検索）"

# ============================================================
# 10. 評価用質問
# ============================================================

questions = [
    "富士山の標高は？",
    "東京タワーの高さは？",
    "東京スカイツリーの高さは？",
    "日本の首都は？",
    "筑波山の標高は？",
    "太宰治の代表作は？",
    "モンブランの麓の町は？",
    "100m走の世界記録は？",
    "マイケルジャクソンの代表作は？",
    "うちに遊びに来る猫の名前は？"
]

# ============================================================
# 11. RAGなし / Fallback付きRAGの比較
# ============================================================

print("\n" + "=" * 70)
print("Embedding RAG評価開始")
print("Embedding model:", EMBEDDING_MODEL)
print("Embedding dimensions:", document_vectors.shape[1])
print("PROMPT_TYPE:", PROMPT_TYPE)
print("TOP_K:", TOP_K)
print("MIN_SCORE:", MIN_SCORE)
print("=" * 70)

for question in questions:

    print("=" * 70)
    print("質問:", question)

    # RAGなし
    answer_no_rag, _, _ = ask_sft(
        question,
        use_rag=False
    )

    print("\n【RAGなし】")
    print(answer_no_rag)

    # Fallback付きEmbedding RAG
    answer_rag, retrieved, mode = ask_sft(
        question,
        use_rag=True,
        prompt_type=PROMPT_TYPE
    )

    print("\n【検索された文章】")

    if retrieved:
        for doc, score in retrieved:
            print(f"score={score:.3f}: {doc}")
    else:
        print("該当する文書なし")

    print("\n【回答方式】")
    print(mode)

    print("\n【RAGあり（Embedding + Fallback）】")
    print(answer_rag)

print("\n全質問の評価終了")


##【STEP7】Word Embeddingの性質　類似疑念の検索　埋め込み空間内の各トークンの類似性を調べます。

##【STEP7-0】ライブラリの呼び出し

In [ ]:
# @title
!pip -q install gguf sentencepiece numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.5/118.5 kB 11.5 MB/s eta 0:00:00


##【STEP7-1】ベクトル空間内で類似のtokenのリストアップ　　入力したトークンに類似したトークン上位１００個を列挙します。

In [ ]:
# @title

import os
import numpy as np
from gguf import GGUFReader
from gguf.quants import dequantize

# ============================================================
# 1. 基本設定
# ============================================================

MODEL_PATH = "/content/nanogpt-670m-sft500-q4.gguf"

TOP_K = 100

# ============================================================
# 2. GGUFモデルを読み込む
# ============================================================

if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(
        f"GGUFファイルが見つかりません: {MODEL_PATH}"
    )

reader = GGUFReader(MODEL_PATH)

print("GGUFモデル読み込み完了")

# ============================================================
# 3. GGUFからトークン一覧を取得
# ============================================================

field = reader.get_field("tokenizer.ggml.tokens")

if field is None:
    raise ValueError(
        "GGUFにtokenizer.ggml.tokensがありません"
    )

tokens = [
    bytes(field.parts[i].tolist()).decode(
        "utf-8", errors="replace"
    )
    for i in field.data
]

VOCAB_SIZE = len(tokens)

token_to_id = {
    token: i
    for i, token in enumerate(tokens)
}

print("語彙数:", VOCAB_SIZE)

# ============================================================
# 4. 埋め込み行列を取り出す
# ============================================================

tensor = next(
    (
        t for t in reader.tensors
        if t.name == "token_embd.weight"
    ),
    None
)

if tensor is None:
    raise ValueError(
        "token_embd.weightが見つかりません"
    )

print("テンソル形状:", tensor.shape)
print("量子化形式:", tensor.tensor_type)

# Q6_K量子化を復元
W = dequantize(
    tensor.data,
    tensor.tensor_type
).astype(np.float32)

# ============================================================
# 5. 行列の形状を調整
# ============================================================

# 目標形状: (語彙数, 埋め込み次元)

if W.ndim != 2:
    raise ValueError(
        f"埋め込み行列が2次元ではありません: {W.shape}"
    )

if W.shape[0] == VOCAB_SIZE:
    pass

elif W.shape[1] == VOCAB_SIZE:
    W = W.T.copy()

else:
    raise ValueError(
        f"埋め込み行列の形状が不正です: {W.shape}"
    )

print("埋め込み行列:", W.shape)

if not np.isfinite(W).all():
    raise ValueError(
        "埋め込み行列にNaNまたはInfが存在します"
    )

# ============================================================
# 6. 平均方向ベクトルを計算
# ============================================================

# 全トークンの平均ベクトル
mean_vector = W.mean(
    axis=0,
    dtype=np.float64
)

mean_norm = np.linalg.norm(mean_vector)

if mean_norm < 1e-12:
    raise ValueError(
        "平均ベクトルのノルムがゼロです"
    )

# 平均方向の単位ベクトル
u = (mean_vector / mean_norm).astype(np.float32)

print("\n平均ベクトルのノルム:", mean_norm)

# ============================================================
# 7. 平均方向の射影成分を除去
# ============================================================

# 各トークンの平均方向への射影量
projection = W @ u

# 平均方向の成分を除去
W_projected = W - projection[:, None] * u[None, :]

print("平均方向の射影除去完了")

# ============================================================
# 8. ベクトルを正規化
# ============================================================

def normalize(X):

    norms = np.linalg.norm(
        X,
        axis=1,
        keepdims=True
    )

    return X / np.maximum(norms, 1e-12)

# 射影除去後の埋め込み
W_normalized = normalize(W_projected)

print("コサイン類似度計算の準備完了")

# ============================================================
# 9. 類似トークン検索
# ============================================================

def find_similar_tokens(token, top_k=TOP_K):

    # --------------------------------------------------------
    # 入力トークンのIDを調べる
    # --------------------------------------------------------

    candidates = [
        token,
        "▁" + token
    ]

    token_id = None

    for candidate in candidates:

        if candidate in token_to_id:

            token_id = token_to_id[candidate]
            break

    if token_id is None:

        print(
            "\n単一トークンとして登録されていません:",
            token
        )

        return

    # --------------------------------------------------------
    # 入力トークンの埋め込みベクトル
    # --------------------------------------------------------

    query_vector = W_normalized[token_id]

    # --------------------------------------------------------
    # 全トークンとのコサイン類似度
    # --------------------------------------------------------

    similarities = W_normalized @ query_vector

    # 自分自身を除外
    similarities[token_id] = -np.inf

    # --------------------------------------------------------
    # 類似度上位TOP_Kを取得
    # --------------------------------------------------------

    k = min(top_k, VOCAB_SIZE - 1)

    indices = np.argsort(similarities)[::-1][:k]

    # --------------------------------------------------------
    # 検索結果表示
    # --------------------------------------------------------

    print("\n" + "=" * 75)

    print("入力トークン:", repr(tokens[token_id]))
    print("Token ID:", token_id)

    print("検索方法: 平均方向の射影除去 + コサイン類似度")

    print("\n類似トークン TOP", k)

    print("-" * 75)

    print(
        f"{'Rank':<6}"
        f"{'Token ID':<12}"
        f"{'Cosine':<12}"
        f"Token"
    )

    print("-" * 75)

    for rank, idx in enumerate(indices, 1):

        print(
            f"{rank:<6}"
            f"{idx:<12}"
            f"{similarities[idx]:<12.4f}"
            f"{repr(tokens[idx])}"
        )

    print("=" * 75)

# ============================================================
# 10. 対話的に検索
# ============================================================

print("\n検索準備完了")
print("トークンを入力してください")
print("終了する場合は exit と入力してください")

while True:

    token = input(
        "\n検索するトークン（exitで終了）: "
    ).strip()

    if token.lower() == "exit":
        print("検索終了")
        break

    if not token:
        continue

    find_similar_tokens(
        token,
        top_k=TOP_K
    )


##【付録】環境のバックアップ。　各自のGoogle DriveにLLMをバックアップアップします。

In [ ]:
# @title
import os
import shutil
from pathlib import Path
from google.colab import drive

# ============================================================
# 1. Google Driveをマウント
# ============================================================

drive.mount("/content/drive")

# ============================================================
# 2. 保存先
# ============================================================

SOURCE_DIR = Path("/content")
BACKUP_DIR = Path("/content/drive/MyDrive/Class_3Backup")

BACKUP_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# 3. 保存対象
# ============================================================

# Google Driveのマウントフォルダは除外
EXCLUDE_DIRS = {
    "drive",
    "sample_data",
    ".cache",
    "__pycache__"
}

# ============================================================
# 4. ファイルをコピー
# ============================================================

file_count = 0

for root, dirs, files in os.walk(SOURCE_DIR):

    dirs[:] = [
        d for d in dirs
        if d not in EXCLUDE_DIRS
    ]

    for filename in files:

        src = Path(root) / filename
        relative_path = src.relative_to(SOURCE_DIR)
        dst = BACKUP_DIR / relative_path

        dst.parent.mkdir(parents=True, exist_ok=True)

        shutil.copy2(src, dst)

        file_count += 1

        print("保存:", relative_path)

print("=" * 60)
print("バックアップ完了")
print("保存ファイル数:", file_count)
print("保存先:", BACKUP_DIR)

##【付録】すでに保存したLLMをGoogle Driveから読み込みます。

In [ ]:
# @title
from pathlib import Path
import shutil

BACKUP_DIR = Path("/content/drive/MyDrive/Class_3Backup")
RESTORE_DIR = Path("/content")

for src in BACKUP_DIR.rglob("*"):

    if src.is_file():

        relative_path = src.relative_to(BACKUP_DIR)
        dst = RESTORE_DIR / relative_path

        dst.parent.mkdir(parents=True, exist_ok=True)

        shutil.copy2(src, dst)

print("ファイルの復元完了")

ファイルの復元完了
